# DenseNet-121 (TorchXRayVision) for Type 2 diabetes detection from chest X-rays

**Final model:** DenseNet-121 pretrained on chest X-rays (`densenet121-res224-all`), fine-tuned to predict T2D
from a single frontal chest radiograph. Learning rate **3e-4** for the new output layer and **3e-5** for the pretrained layers.

**Evaluation:** 5-fold cross-validation grouped by patient, run with 3 seeds, on the development set (80%).
A locked, patient-independent test set (20%) is scored once at the end.

| | ROC-AUC (95% CI) | Notes |
|---|---|---|
| Cross-validation (n = 758) | 0.624 (0.579–0.664) | AP 0.445 |
| **Held-out test (n = 190)** | **0.703 (0.616–0.781)** | Sens 0.727, Spec 0.519, F1 0.500 at threshold 0.488 |

The same code lives as `.py` files in `main_model/densenet_t2d/`; this notebook writes a copy so it runs standalone in Colab.

### Before running
1. *Runtime → Change runtime type → T4 GPU*.
2. In Google Drive, `My Drive/T2D_CXR/` should contain `Image Folder/` (the cleaned frontal X-rays) and the labels CSV.
   If you already ran this model, results in `T2D_CXR/results/` are reused, not retrained.
3. *Runtime → Run all*.

## 1. Settings

In [ ]:
DRIVE_DIR  = "/content/drive/MyDrive/T2D_CXR"
IMAGE_DIR  = DRIVE_DIR + "/Image Folder"                    # cleaned frontal X-rays
LABELS_CSV = DRIVE_DIR + "/Clinical_data_with_SDI.csv"      # needs image_name and T2D columns

# Name of the experiment folder under results/experiments/.
# "05_densenet_lr3e-4_final" is the run already completed, so it's reused instead of retrained.
EXPERIMENT = "05_densenet_lr3e-4_final"

## 2. Connect Drive, check the GPU, install packages

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab - skipping Drive mount")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")
for p in [DRIVE_DIR, IMAGE_DIR, LABELS_CSV]:
    print(("OK      " if os.path.exists(p) else "MISSING ") + p)

In [ ]:
!pip install -q torchxrayvision pyyaml opencv-python-headless

## 3. Pipeline code
Writes the training code to `/content/angelo_densenet121`. Run these cells; no need to edit them.
- `splits.py`: patient-grouped, stratified train/test split and 5 CV folds
- `data.py`: image loading, aspect-preserving resize, train-only augmentation
- `models.py`: the DenseNet-121 model
- `metrics.py`: ROC-AUC, AP, sensitivity/specificity, bootstrap confidence intervals
- `train_cv.py`: cross-validated training with early stopping
- `final_test.py`: one-time scoring of the held-out test set

In [ ]:
import os
os.makedirs("/content/angelo_densenet121/densenet_t2d", exist_ok=True)
os.makedirs("/content/angelo_densenet121/configs", exist_ok=True)
open("/content/angelo_densenet121/densenet_t2d/__init__.py", "w").close()

In [ ]:
%%writefile /content/angelo_densenet121/densenet_t2d/splits.py
"""Leakage-safe data splits.

Creates ONE split file that every experiment reads, so all models are compared
on exactly the same images:

    split == "test"  -> locked hold-out set. Touched once, by final_test.py.
    split == "dev"   -> used for model development, with a `fold` column (0..K-1)
                        from StratifiedGroupKFold grouped by patient.

Usage
-----
    # Fresh patient-grouped split (20% test, 5 CV folds on the rest)
    python -m densenet_t2d.splits --csv patient.csv --image-dir "Image Folder" \
        --out splits/splits.csv

    # Re-use the team's existing hold-out test set (malek_and_angelo/malek)
    python -m densenet_t2d.splits --csv patient.csv --image-dir "Image Folder" \
        --out splits/splits.csv \
        --team-split ../../malek_and_angelo/malek/train_val_test_split.csv
"""
from __future__ import annotations

import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

PATIENT_REGEX = r"^(CXR\d+)"


def extract_patient_id(image_name: str, regex: str = PATIENT_REGEX) -> str:
    match = re.match(regex, str(image_name))
    return match.group(1) if match else str(image_name)


def load_labels(csv_file, image_dir, label_col="T2D", patient_col=None, regex=PATIENT_REGEX):
    df = pd.read_csv(csv_file)
    for col in ("image_name", label_col):
        if col not in df.columns:
            raise ValueError(f"Missing column: {col}")
    df = df.dropna(subset=["image_name", label_col]).copy()
    df["image_name"] = df["image_name"].astype(str)
    if not df[label_col].isin([0, 1]).all():
        raise ValueError(f"{label_col} labels must be 0 or 1")
    df["label"] = df[label_col].astype(int)

    if df["image_name"].duplicated().any():
        raise ValueError("Duplicate image filenames found")

    image_dir = Path(image_dir)
    missing = [n for n in df["image_name"] if not (image_dir / n).is_file()]
    if missing:
        raise FileNotFoundError(f"{len(missing)} images missing, e.g. {missing[:5]}")

    if patient_col and patient_col in df.columns:
        df["patient_id"] = df[patient_col].astype(str)
    else:
        df["patient_id"] = df["image_name"].map(lambda n: extract_patient_id(n, regex))

    # A patient must have one label; otherwise grouping is ambiguous.
    conflicting = df.groupby("patient_id")["label"].nunique()
    if (conflicting > 1).any():
        raise ValueError(f"Patients with conflicting labels: {list(conflicting[conflicting > 1].index[:5])}")

    return df[["image_name", "patient_id", "label"]].reset_index(drop=True)


def assign_folds(dev: pd.DataFrame, n_folds: int, seed: int) -> pd.Series:
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = pd.Series(-1, index=dev.index, dtype=int)
    for k, (_, idx) in enumerate(sgkf.split(dev, dev["label"], groups=dev["patient_id"])):
        folds.iloc[idx] = k
    return folds


def make_splits(df, test_frac=0.2, n_folds=5, seed=42, team_split=None):
    df = df.copy()
    if team_split is not None:
        team = pd.read_csv(team_split)
        test_names = set(team.loc[team["split"].str.upper() == "TEST", "image_name"].astype(str))
        test_patients = set(df.loc[df["image_name"].isin(test_names), "patient_id"])
        df["split"] = np.where(df["patient_id"].isin(test_patients), "test", "dev")
        unmatched = len(test_names - set(df["image_name"]))
        if unmatched:
            print(f"[splits] note: {unmatched} team test images are not in your label CSV")
    else:
        # 1/test_frac folds, take one fold as test -> stratified AND patient-grouped
        n = max(2, round(1 / test_frac))
        sgkf = StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=seed)
        _, test_idx = next(sgkf.split(df, df["label"], groups=df["patient_id"]))
        df["split"] = "dev"
        df.loc[df.index[test_idx], "split"] = "test"

    df["fold"] = -1
    dev_mask = df["split"] == "dev"
    df.loc[dev_mask, "fold"] = assign_folds(df[dev_mask], n_folds, seed).values
    check_splits(df)
    return df


def check_splits(df: pd.DataFrame) -> None:
    """Hard failures for anything that would leak information."""
    test_p = set(df.loc[df["split"] == "test", "patient_id"])
    dev_p = set(df.loc[df["split"] == "dev", "patient_id"])
    assert not (test_p & dev_p), "Patient appears in both dev and test"
    per_patient_folds = df[df["split"] == "dev"].groupby("patient_id")["fold"].nunique()
    assert (per_patient_folds == 1).all(), "Patient appears in more than one CV fold"
    assert (df.loc[df["split"] == "dev", "fold"] >= 0).all()


def summarize(df: pd.DataFrame) -> str:
    lines = [f"Images: {len(df)}  Patients: {df['patient_id'].nunique()}  "
             f"Prevalence: {df['label'].mean():.3f}"]
    multi = (df["patient_id"].value_counts() > 1).sum()
    lines.append(f"Patients with >1 image: {multi}")
    for name, g in df.groupby("split"):
        lines.append(f"  {name:4s}: n={len(g):4d}  pos={g['label'].sum():4d}  prev={g['label'].mean():.3f}")
    for k, g in df[df["split"] == "dev"].groupby("fold"):
        lines.append(f"    fold {k}: n={len(g):4d}  pos={g['label'].sum():4d}  prev={g['label'].mean():.3f}")
    return "\n".join(lines)


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--csv", required=True)
    p.add_argument("--image-dir", required=True)
    p.add_argument("--out", default="splits/splits.csv")
    p.add_argument("--label-col", default="T2D")
    p.add_argument("--patient-col", default=None, help="Use this column as patient id if present")
    p.add_argument("--patient-regex", default=PATIENT_REGEX)
    p.add_argument("--test-frac", type=float, default=0.2)
    p.add_argument("--n-folds", type=int, default=5)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--team-split", default=None, help="Re-use TEST rows of an existing split CSV")
    p.add_argument("--force", action="store_true", help="Overwrite an existing split file")
    a = p.parse_args()

    out = Path(a.out)
    if out.exists() and not a.force:
        raise SystemExit(f"{out} already exists. Splits should be created once; pass --force to overwrite.")

    df = load_labels(a.csv, a.image_dir, a.label_col, a.patient_col, a.patient_regex)
    df = make_splits(df, a.test_frac, a.n_folds, a.seed, a.team_split)
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out, index=False)
    print(summarize(df))
    print(f"\nSaved {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/angelo_densenet121/densenet_t2d/data.py
"""Image loading, deterministic preprocessing, and train-only augmentation.

Deterministic preprocessing (identical for train/val/test, cached to disk):
    grayscale -> robust 16-bit handling -> optional CLAHE -> aspect-preserving
    resize of the long side -> zero-pad to a square canvas.

Random augmentation (training only):
    small rotation / translation / scale, brightness & contrast jitter.
    NO horizontal flips: heart and aortic knob are on the left in a frontal CXR,
    and those are plausible T2D-related signals.

Normalization lives inside each model (see models.py), so the dataset always
returns 1xHxW float tensors in [0, 1].
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms import v2

try:
    import cv2
except ImportError:  # CLAHE becomes unavailable, everything else works
    cv2 = None


def load_grayscale(path: Path) -> np.ndarray:
    """Return a uint8 HxW array. 16-bit PNGs are rescaled with 0.5/99.5 percentiles."""
    with Image.open(path) as img:
        if img.mode in ("I;16", "I;16B", "I;16L", "I", "F"):
            arr = np.asarray(img, dtype=np.float32)
            lo, hi = np.percentile(arr, [0.5, 99.5])
            arr = np.clip((arr - lo) / max(hi - lo, 1e-6), 0, 1) * 255
            return arr.astype(np.uint8)
        return np.asarray(img.convert("L"), dtype=np.uint8)


def letterbox(arr: np.ndarray, size: int) -> np.ndarray:
    h, w = arr.shape
    scale = size / max(h, w)
    nh, nw = max(1, round(h * scale)), max(1, round(w * scale))
    resized = np.asarray(Image.fromarray(arr).resize((nw, nh), Image.Resampling.BILINEAR))
    canvas = np.zeros((size, size), dtype=np.uint8)
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas[top:top + nh, left:left + nw] = resized
    return canvas


def preprocess(path: Path, size: int, clahe: bool, clahe_clip: float = 2.0) -> np.ndarray:
    arr = load_grayscale(path)
    if clahe:
        if cv2 is None:
            raise ImportError("CLAHE requested but opencv-python is not installed")
        arr = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8)).apply(arr)
    return letterbox(arr, size)


def build_cache(names, image_dir, size, clahe, cache_dir) -> np.ndarray:
    """Preprocess every image once and store uint8 arrays (N, H, W) on disk."""
    key = json.dumps({"names": sorted(names), "dir": str(Path(image_dir).resolve()),
                      "size": size, "clahe": clahe}, sort_keys=True)
    digest = hashlib.sha1(key.encode()).hexdigest()[:12]
    cache_file = Path(cache_dir) / f"images_{size}_{'clahe' if clahe else 'raw'}_{digest}.npz"
    order = sorted(names)
    if cache_file.exists():
        stored = np.load(cache_file, allow_pickle=False)
        arrays = stored["images"]
    else:
        Path(cache_dir).mkdir(parents=True, exist_ok=True)
        print(f"[data] preprocessing {len(order)} images -> {cache_file.name}")
        arrays = np.stack([preprocess(Path(image_dir) / n, size, clahe) for n in order])
        np.savez_compressed(cache_file, images=arrays)
    index = {n: i for i, n in enumerate(order)}
    return arrays[[index[n] for n in names]]


def make_train_transform(cfg: dict):
    aug = cfg.get("augment", {})
    if not aug.get("enabled", True):
        return None
    return v2.Compose([
        v2.RandomAffine(
            degrees=aug.get("degrees", 7),
            translate=(aug.get("translate", 0.06),) * 2,
            scale=(1 - aug.get("scale", 0.08), 1 + aug.get("scale", 0.08)),
        ),
        v2.ColorJitter(brightness=aug.get("brightness", 0.15), contrast=aug.get("contrast", 0.15)),
    ])


class CXRDataset(Dataset):
    def __init__(self, images_uint8: np.ndarray, labels, transform=None):
        self.images = torch.from_numpy(np.ascontiguousarray(images_uint8)).unsqueeze(1)  # (N,1,H,W) uint8
        self.labels = torch.tensor(np.asarray(labels), dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        x = self.images[i].float().div_(255.0)
        if self.transform is not None:
            x = self.transform(x).clamp_(0, 1)
        return x, self.labels[i]


def load_split_table(splits_csv) -> pd.DataFrame:
    df = pd.read_csv(splits_csv)
    need = {"image_name", "patient_id", "label", "split", "fold"}
    if not need <= set(df.columns):
        raise ValueError(f"{splits_csv} missing columns {need - set(df.columns)}")
    return df


In [ ]:
%%writefile /content/angelo_densenet121/densenet_t2d/models.py
"""DenseNet-121 pretrained on chest X-rays (TorchXRayVision), with a new single-output T2D head."""
import torch
import torch.nn as nn
import torch.nn.functional as F


class DenseNet121XRV(nn.Module):
    def __init__(self, weights="densenet121-res224-all", dropout=0.3):
        super().__init__()
        import torchxrayvision as xrv
        net = xrv.models.DenseNet(weights=weights)       # DenseNet-121 + chest X-ray pretrained weights
        self.features = net.features                      # keep the pretrained feature layers
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(1024, 1))  # new T2D output (logit)

    def forward(self, x):
        x = x * 2048.0 - 1024.0                           # [0,1] -> [-1024,1024], the range XRV expects
        f = F.relu(self.features(x), inplace=True)
        return self.head(torch.flatten(F.adaptive_avg_pool2d(f, 1), 1))

    def param_groups(self, lr, backbone_lr_mult=1.0):
        """Head trains at lr; pretrained layers at lr * backbone_lr_mult."""
        head_ids = {id(p) for p in self.head.parameters()}
        backbone = [p for p in self.parameters() if id(p) not in head_ids]
        return [{"params": list(self.head.parameters()), "lr": lr},
                {"params": backbone, "lr": lr * backbone_lr_mult}]


REGISTRY = {"densenet121_xrv": DenseNet121XRV}


def build_model(cfg):
    m = dict(cfg["model"])
    return REGISTRY[m.pop("name")](**m)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
%%writefile /content/angelo_densenet121/densenet_t2d/metrics.py
"""Metrics, thresholds, and bootstrap confidence intervals."""
from __future__ import annotations

import numpy as np
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score, roc_curve


def threshold_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if tp + fn else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    prec = tp / (tp + fp) if tp + fp else 0.0
    f1 = 2 * prec * sens / (prec + sens) if prec + sens else 0.0
    return {"threshold": float(thr), "accuracy": (tp + tn) / len(y), "sensitivity": sens,
            "specificity": spec, "precision": prec, "f1": f1,
            "balanced_accuracy": (sens + spec) / 2,
            "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]]}


def ranking_metrics(y, p):
    return {"roc_auc": float(roc_auc_score(y, p)), "average_precision": float(average_precision_score(y, p)),
            "brier": float(np.mean((p - y) ** 2)), "n": int(len(y)), "n_pos": int(y.sum())}


def choose_threshold(y, p, method="youden", target_sensitivity=0.8):
    """Pick a threshold on DEVELOPMENT predictions only; then freeze it for test."""
    fpr, tpr, thr = roc_curve(y, p)
    thr = np.clip(thr, 0, 1)
    if method == "youden":
        return float(thr[np.argmax(tpr - fpr)])
    if method == "sensitivity":
        ok = np.where(tpr >= target_sensitivity)[0]
        return float(thr[ok[0]])  # highest threshold reaching the target sensitivity
    if method == "fixed":
        return 0.5
    raise ValueError(method)


def bootstrap_ci(y, p, fn, n_boot=2000, seed=0, alpha=0.05):
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    stats = []
    for _ in range(n_boot):  # stratified resampling keeps prevalence fixed
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        stats.append(fn(y[idx], p[idx]))
    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)


def paired_bootstrap_auc_diff(y, p_a, p_b, n_boot=2000, seed=0):
    """AUC(b) - AUC(a) on the same images, with 95% CI and a two-sided p-value."""
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        diffs.append(roc_auc_score(y[idx], p_b[idx]) - roc_auc_score(y[idx], p_a[idx]))
    diffs = np.asarray(diffs)
    observed = roc_auc_score(y, p_b) - roc_auc_score(y, p_a)
    p_val = min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"auc_diff": float(observed), "ci95": [float(lo), float(hi)], "p_value": float(p_val)}


def full_report(y, p, thresholds: dict, n_boot=2000):
    y, p = np.asarray(y).astype(int), np.asarray(p, dtype=float)
    out = ranking_metrics(y, p)
    out["roc_auc_ci95"] = bootstrap_ci(y, p, roc_auc_score, n_boot)
    out["average_precision_ci95"] = bootstrap_ci(y, p, average_precision_score, n_boot)
    out["prevalence"] = float(y.mean())
    out["at_threshold"] = {name: threshold_metrics(y, p, t) for name, t in thresholds.items()}
    return out


In [ ]:
%%writefile /content/angelo_densenet121/densenet_t2d/train_cv.py
"""K-fold cross-validated training on the DEV split only (the test split is never loaded).

For each outer fold k (and each seed):
  1. outer-val = dev images in fold k        -> scored once, after training
  2. the rest is split again, by patient,
     into inner-train / inner-val            -> inner-val drives early stopping
  3. best-epoch weights predict outer-val    -> out-of-fold (OOF) predictions

Because the images that pick the epoch are never the images that are scored,
the OOF metrics are an honest estimate. Compare experiments on these numbers.

    python -m densenet_t2d.train_cv --config configs/densenet121_xrv_final.yaml
    python -m densenet_t2d.train_cv --config configs/densenet121_xrv_final.yaml --set train.lr=1e-3 --name densenet_lr1e-3
"""
from __future__ import annotations

import argparse
import copy
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yaml
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader

from .data import CXRDataset, build_cache, load_split_table, make_train_transform
from .metrics import choose_threshold, full_report
from .models import build_model, count_params

ROOT = Path(__file__).resolve().parents[1]  # main_model


# ----------------------------------------------------------------------------- config
def deep_merge(base, over):
    out = copy.deepcopy(base)
    for k, v in over.items():
        out[k] = deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


def load_config(path, overrides=()):
    """YAML config; a `base:` key (relative path) is loaded first and overridden."""
    path = Path(path)
    cfg = yaml.safe_load(path.read_text())
    if "base" in cfg:
        base = load_config(path.parent / cfg.pop("base"))
        if "model" in cfg:  # a model block replaces the base one (different models take different args)
            base.pop("model", None)
        cfg = deep_merge(base, cfg)
    for item in overrides:
        key, val = item.split("=", 1)
        node = cfg
        *parents, leaf = key.split(".")
        for p in parents:
            node = node.setdefault(p, {})
        node[leaf] = yaml.safe_load(val)
    return cfg


def resolve(path):
    p = Path(path)
    return p if p.is_absolute() else ROOT / p


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ----------------------------------------------------------------------------- training
def inner_split(df, frac, seed):
    n = max(2, round(1 / frac))
    sgkf = StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=seed)
    tr, va = next(sgkf.split(df, df["label"], groups=df["patient_id"]))
    return df.iloc[tr], df.iloc[va]


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    out = []
    for x, _ in loader:
        out.append(torch.sigmoid(model(x.to(device))).squeeze(1).float().cpu())
    return torch.cat(out).numpy()


def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device).unsqueeze(1)
            total += criterion(model(x), y).item() * len(y)
            n += len(y)
    return total / n


def make_scheduler(opt, epochs, warmup):
    def f(e):
        if e < warmup:
            return (e + 1) / warmup
        prog = (e - warmup) / max(1, epochs - warmup)
        return 0.5 * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


def train_one(cfg, images, tr_df, va_df, device, seed, log_prefix=""):
    """Train on tr_df, early-stop on va_df. Returns best model + per-epoch history."""
    t = cfg["train"]
    set_seed(seed)
    g = torch.Generator().manual_seed(seed)

    tr_ds = CXRDataset(images[tr_df["row"].values], tr_df["label"].values, make_train_transform(cfg))
    va_ds = CXRDataset(images[va_df["row"].values], va_df["label"].values)
    nw = t.get("num_workers", 0)
    tr_dl = DataLoader(tr_ds, batch_size=t["batch_size"], shuffle=True, drop_last=True, generator=g, num_workers=nw)
    va_dl = DataLoader(va_ds, batch_size=64, shuffle=False, num_workers=nw)

    model = build_model(cfg).to(device)
    pos = tr_df["label"].sum()
    pw = t.get("pos_weight", "auto")
    pw = (len(tr_df) - pos) / pos if pw == "auto" else float(pw)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([float(pw)], dtype=torch.float32, device=device))
    smooth = float(t.get("label_smoothing", 0.0))

    opt = torch.optim.AdamW(model.param_groups(float(t["lr"]), float(t.get("backbone_lr_mult", 1.0))),
                            weight_decay=float(t.get("weight_decay", 1e-4)))
    sched = make_scheduler(opt, t["epochs"], t.get("warmup_epochs", 3))

    es = t.get("early_stopping", {})
    monitor, patience = es.get("monitor", "val_loss"), es.get("patience", 10)
    best_score, best_state, best_epoch, bad = -np.inf, None, 0, 0
    history = []

    for epoch in range(1, t["epochs"] + 1):
        model.train()
        run, n = 0.0, 0
        for x, y in tr_dl:
            x, y = x.to(device), y.to(device).unsqueeze(1)
            y_s = y * (1 - smooth) + 0.5 * smooth
            loss = criterion(model(x), y_s)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            if t.get("grad_clip"):
                nn.utils.clip_grad_norm_(model.parameters(), t["grad_clip"])
            opt.step()
            run += loss.item() * len(y)
            n += len(y)
        sched.step()

        val_loss = evaluate_loss(model, va_dl, criterion, device)
        val_auc = roc_auc_score(va_df["label"].values, predict(model, va_dl, device))
        history.append({"epoch": epoch, "train_loss": run / n, "inner_val_loss": val_loss,
                        "inner_val_auc": val_auc, "lr": opt.param_groups[0]["lr"]})
        score = -val_loss if monitor == "val_loss" else val_auc
        improved = score > best_score + 1e-4
        if improved:
            best_score, best_state, best_epoch, bad = score, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            bad += 1
        print(f"{log_prefix} ep {epoch:3d} | train {run / n:.4f} | inner-val loss {val_loss:.4f} "
              f"auc {val_auc:.4f}{' *' if improved else ''}", flush=True)
        if bad >= patience:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch, pw


def plot_curves(hist: pd.DataFrame, out_png: Path):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    runs = list(hist.groupby(["fold", "seed"]))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for (k, s), h in runs:
        axes[0].plot(h["epoch"], h["train_loss"], color="tab:blue", alpha=0.5)
        axes[0].plot(h["epoch"], h["inner_val_loss"], color="tab:orange", alpha=0.5)
        axes[1].plot(h["epoch"], h["inner_val_auc"], color="tab:green", alpha=0.5)
    axes[0].set(title="Loss (blue=train, orange=inner-val)", xlabel="epoch")
    axes[1].set(title="Inner-val ROC-AUC", xlabel="epoch")
    for ax in axes:
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_png, dpi=120)
    plt.close(fig)


# ----------------------------------------------------------------------------- main
def run(cfg, name=None, max_folds=None, overwrite=False):
    name = name or cfg["name"]
    out_dir = resolve(cfg.get("results_dir", "results/experiments")) / name
    if (out_dir / "summary.json").exists() and not overwrite:
        raise SystemExit(f"{out_dir} already has results. Use a new --name, or --overwrite.")
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))

    d = cfg["data"]
    table = load_split_table(resolve(d["splits_csv"]))
    names = table["image_name"].tolist()
    images = build_cache(names, resolve(d["image_dir"]), d["image_size"], d.get("clahe", False),
                         resolve(d.get("cache_dir", "results/cache")))
    table["row"] = np.arange(len(table))
    dev = table[table["split"] == "dev"].reset_index(drop=True)  # test rows are never used here

    device = get_device()
    seeds = cfg["train"].get("seeds", [42])
    folds = sorted(dev["fold"].unique())[: max_folds or None]
    print(f"[cv] {name} | device={device} | dev={len(dev)} | folds={folds} | seeds={seeds} | "
          f"params={count_params(build_model(cfg)):,}")

    oof_rows, histories, fold_info = [], [], []
    t0 = time.time()
    for k in folds:
        outer = dev[dev["fold"] == k]
        pool = dev[dev["fold"] != k]
        for s in seeds:
            tr, va = inner_split(pool, cfg["train"].get("early_stopping", {}).get("inner_val_frac", 0.15), seed=s + k)
            model, hist, best_ep, pw = train_one(cfg, images, tr, va, device, seed=s, log_prefix=f"[fold {k} seed {s}]")
            outer_dl = DataLoader(CXRDataset(images[outer["row"].values], outer["label"].values), batch_size=64)
            p = predict(model, outer_dl, device)
            fold_auc = roc_auc_score(outer["label"], p)
            print(f"[fold {k} seed {s}] best epoch {best_ep} | OUTER-fold AUC {fold_auc:.4f}")
            torch.save({"state_dict": model.state_dict(), "config": cfg, "fold": int(k), "seed": int(s),
                        "best_epoch": best_ep, "pos_weight": pw}, out_dir / f"fold{k}_seed{s}.pt")
            hist["fold"], hist["seed"] = k, s
            histories.append(hist)
            fold_info.append({"fold": int(k), "seed": int(s), "best_epoch": best_ep, "outer_auc": fold_auc,
                              "n_train": len(tr), "n_inner_val": len(va), "n_outer": len(outer)})
            oof_rows.append(pd.DataFrame({"image_name": outer["image_name"].values, "patient_id": outer["patient_id"].values,
                                          "label": outer["label"].values, "fold": k, "seed": s, "prob": p}))

    hist = pd.concat(histories)
    hist.to_csv(out_dir / "history.csv", index=False)
    plot_curves(hist, out_dir / "learning_curves.png")
    oof_all = pd.concat(oof_rows)
    oof_all.to_csv(out_dir / "oof_per_seed.csv", index=False)
    # Average seeds -> one prediction per image (a small seed-ensemble)
    oof = oof_all.groupby(["image_name", "patient_id", "label", "fold"], as_index=False)["prob"].mean()
    oof.to_csv(out_dir / "oof.csv", index=False)

    ev = cfg.get("evaluation", {})
    thr = choose_threshold(oof["label"].values, oof["prob"].values, ev.get("threshold_method", "youden"),
                           ev.get("target_sensitivity", 0.8))
    report = full_report(oof["label"].values, oof["prob"].values, {"fixed_0.50": 0.5, "dev_selected": thr})
    fi = pd.DataFrame(fold_info)
    per_fold = oof.groupby("fold").apply(lambda g: roc_auc_score(g["label"], g["prob"]), include_groups=False)
    summary = {
        "experiment": name, "data_used": "dev split only (out-of-fold)", "params": count_params(build_model(cfg)),
        "oof": report,
        "per_fold_auc": {int(k): float(v) for k, v in per_fold.items()},
        "per_fold_auc_mean": float(per_fold.mean()), "per_fold_auc_sd": float(per_fold.std(ddof=1)) if len(per_fold) > 1 else None,
        "best_epochs": fi["best_epoch"].tolist(), "runs": fold_info,
        "selected_threshold": thr,
        "note": "dev_selected threshold metrics are optimistic (threshold chosen on these same OOF predictions). "
                "Its frozen value is what final_test.py applies to the test set.",
        "minutes": round((time.time() - t0) / 60, 1),
    }
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    r = report
    print(f"\n[cv] {name}: OOF AUC {r['roc_auc']:.4f} (95% CI {r['roc_auc_ci95'][0]:.3f}-{r['roc_auc_ci95'][1]:.3f}) | "
          f"AP {r['average_precision']:.4f} | fold AUC {summary['per_fold_auc_mean']:.4f} ± {summary['per_fold_auc_sd'] or 0:.4f}")
    print(f"[cv] results -> {out_dir}")
    return summary


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--config", required=True)
    p.add_argument("--name", default=None, help="Experiment name (defaults to config 'name')")
    p.add_argument("--set", nargs="*", default=[], help="Overrides, e.g. train.lr=1e-3 model.dropout=0.5")
    p.add_argument("--max-folds", type=int, default=None, help="Debug only: run the first N folds")
    p.add_argument("--overwrite", action="store_true", help="Replace an existing experiment's results")
    a = p.parse_args()
    cfg = load_config(a.config, a.set)
    if a.name:
        cfg["name"] = a.name
    run(cfg, a.name, a.max_folds, a.overwrite)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/angelo_densenet121/densenet_t2d/final_test.py
"""One-time evaluation of a FINISHED experiment on the locked test split.

The K fold models from train_cv.py are used as an ensemble (mean probability),
so no retraining and no extra choices are made after looking at the dev data.
The decision threshold is the one frozen from OOF predictions in summary.json.

Every evaluation is appended to results/test_log.csv. Running a second model on
the test set requires --force and is logged: each extra look turns the test set
a little more into a validation set, so do this only for your final 1-2 models.

    python -m densenet_t2d.final_test --experiment densenet121_xrv_final
"""
from __future__ import annotations

import argparse
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

from .data import CXRDataset, build_cache, load_split_table
from .metrics import full_report
from .models import build_model
from .train_cv import get_device, predict, resolve

ROOT = Path(__file__).resolve().parents[1]


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--experiment", required=True, help="Name under results/experiments/ or a path")
    p.add_argument("--force", action="store_true", help="Allow evaluating after the test set was already used")
    a = p.parse_args()

    exp = Path(a.experiment)
    if not exp.exists():
        exp = ROOT / "results" / "experiments" / a.experiment
    summary = json.loads((exp / "summary.json").read_text())
    log_path = ROOT / "results" / "test_log.csv"
    if log_path.exists() and not a.force:
        prev = pd.read_csv(log_path)
        raise SystemExit(f"Test set already evaluated {len(prev)} time(s): {prev['experiment'].tolist()}.\n"
                         f"Re-run with --force only if this is a pre-planned final comparison.")

    ckpts = sorted(exp.glob("fold*_seed*.pt"))
    if not ckpts:
        raise SystemExit(f"No fold checkpoints in {exp}")
    cfg = torch.load(ckpts[0], map_location="cpu", weights_only=False)["config"]
    d = cfg["data"]
    table = load_split_table(resolve(d["splits_csv"]))
    images = build_cache(table["image_name"].tolist(), resolve(d["image_dir"]), d["image_size"],
                         d.get("clahe", False), resolve(d.get("cache_dir", "results/cache")))
    test_mask = (table["split"] == "test").values
    test = table[test_mask].reset_index(drop=True)
    dl = DataLoader(CXRDataset(images[test_mask], test["label"].values), batch_size=64)

    device = get_device()
    probs = []
    for c in ckpts:
        ck = torch.load(c, map_location=device, weights_only=False)
        model = build_model(ck["config"]).to(device)
        model.load_state_dict(ck["state_dict"])
        probs.append(predict(model, dl, device))
    probs = np.stack(probs)
    p_ens = probs.mean(0)

    thr = summary["selected_threshold"]
    report = full_report(test["label"].values, p_ens, {"fixed_0.50": 0.5, "dev_selected": thr})
    report["experiment"] = summary["experiment"]
    report["n_models_in_ensemble"] = len(ckpts)
    report["single_model_test_auc"] = {c.stem: float(roc_auc_score(test["label"], pr)) for c, pr in zip(ckpts, probs)}
    report["dev_oof_auc"] = summary["oof"]["roc_auc"]
    (exp / "test_report.json").write_text(json.dumps(report, indent=2))
    pd.DataFrame({"image_name": test["image_name"], "label": test["label"], "prob": p_ens}).to_csv(
        exp / "test_predictions.csv", index=False)

    entry = pd.DataFrame([{"experiment": summary["experiment"], "timestamp": datetime.now().isoformat(timespec="seconds"),
                           "test_auc": report["roc_auc"], "forced": a.force}])
    log_path.parent.mkdir(parents=True, exist_ok=True)
    entry.to_csv(log_path, mode="a", header=not log_path.exists(), index=False)

    r, t = report, report["at_threshold"]["dev_selected"]
    print(f"TEST  n={r['n']} pos={r['n_pos']}")
    print(f"  ROC-AUC {r['roc_auc']:.4f} (95% CI {r['roc_auc_ci95'][0]:.3f}-{r['roc_auc_ci95'][1]:.3f})   "
          f"[dev OOF was {r['dev_oof_auc']:.4f}]")
    print(f"  AP      {r['average_precision']:.4f} (95% CI {r['average_precision_ci95'][0]:.3f}-{r['average_precision_ci95'][1]:.3f})"
          f"   [prevalence {r['prevalence']:.3f}]")
    print(f"  @thr {thr:.3f} (frozen from dev): sens {t['sensitivity']:.3f} spec {t['specificity']:.3f} "
          f"F1 {t['f1']:.3f} CM {t['confusion_matrix']}")
    print(f"Saved {exp / 'test_report.json'}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/angelo_densenet121/configs/densenet121_xrv_final.yaml
# FINAL MODEL: DenseNet-121 pretrained on chest X-rays (TorchXRayVision densenet121-res224-all),
# fine-tuned for T2D with the lower learning rate (3e-4).
name: densenet121_xrv_final
results_dir: results/experiments

data:
  splits_csv: splits/splits.csv
  image_dir: "Image Folder"        # overridden by the notebook's IMAGE_DIR setting
  image_size: 224
  clahe: false
  cache_dir: results/cache

augment:                           # training images only
  enabled: true
  degrees: 7
  translate: 0.06
  scale: 0.08
  brightness: 0.15
  contrast: 0.15

model:
  name: densenet121_xrv
  weights: densenet121-res224-all
  dropout: 0.3

train:
  epochs: 40
  warmup_epochs: 3
  batch_size: 16
  lr: 0.0003                       # head learning rate (final choice)
  backbone_lr_mult: 0.1            # pretrained layers learn at 3e-5
  weight_decay: 0.0001
  pos_weight: auto                 # negatives / positives in each fold's training data
  label_smoothing: 0.0
  grad_clip: 1.0
  num_workers: 0
  seeds: [0, 1, 2]
  early_stopping:
    inner_val_frac: 0.15
    monitor: val_loss
    patience: 10

evaluation:
  threshold_method: youden
  target_sensitivity: 0.80

## 4. Data: keep cleaned images, link Drive folders, create the split
Keeps only label rows whose image is in `IMAGE_DIR`, which excludes the removed lateral and low-quality images.
An existing `splits/splits.csv` in Drive is reused, so the test set never changes.

In [ ]:
import os, pandas as pd
os.chdir("/content/angelo_densenet121")
for name in ["results", "splits"]:
    os.makedirs(f"{DRIVE_DIR}/{name}", exist_ok=True)
    if os.path.islink(name):
        os.remove(name)
    if not os.path.exists(name):
        os.symlink(f"{DRIVE_DIR}/{name}", name)

CLEAN_CSV = DRIVE_DIR + "/labels_frontal_clean.csv"
labels = pd.read_csv(LABELS_CSV)
present = set(os.listdir(IMAGE_DIR))
clean = labels[labels["image_name"].isin(present)]
clean.to_csv(CLEAN_CSV, index=False)
print(f"Label rows: {len(labels)} -> with a cleaned image: {len(clean)}")

if os.path.exists("splits/splits.csv"):
    s = pd.read_csv("splits/splits.csv")
    print("Using existing split:")
    print(s.groupby("split")["label"].agg(images="count", T2D="sum"))
else:
    !python -m densenet_t2d.splits --csv "{CLEAN_CSV}" --image-dir "{IMAGE_DIR}" --out splits/splits.csv

## 5. Train DenseNet-121 (5-fold CV × 3 seeds)
Skipped automatically if this experiment has already finished. A full run takes roughly 1–2 hours on a T4.

In [ ]:
import os, subprocess
if os.path.exists(f"results/experiments/{EXPERIMENT}/summary.json"):
    print(f"{EXPERIMENT} already trained - reusing saved results.")
else:
    cmd = ["python", "-m", "densenet_t2d.train_cv", "--config", "configs/densenet121_xrv_final.yaml",
           "--name", EXPERIMENT, "--overwrite", "--set", f"data.image_dir={IMAGE_DIR}", "train.num_workers=2"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError("Training failed - see the error above")

## 6. Cross-validation results (development set)

In [ ]:
import json, pandas as pd
s = json.load(open(f"results/experiments/{EXPERIMENT}/summary.json"))
o = s["oof"]
print(f"Images: {o['n']}  (T2D: {o['n_pos']}, prevalence {o['prevalence']:.3f})")
print(f"ROC-AUC: {o['roc_auc']:.3f}  (95% CI {o['roc_auc_ci95'][0]:.3f}-{o['roc_auc_ci95'][1]:.3f})")
print(f"Average precision: {o['average_precision']:.3f}  (95% CI {o['average_precision_ci95'][0]:.3f}-{o['average_precision_ci95'][1]:.3f})")
print(f"Per-fold ROC-AUC: {s['per_fold_auc_mean']:.3f} ± {s['per_fold_auc_sd']:.3f}")
print(f"Selected threshold (Youden, frozen for test): {s['selected_threshold']:.3f}\n")

rows = []
for name, m in o["at_threshold"].items():
    (tn, fp), (fn, tp) = m["confusion_matrix"]
    rows.append({"threshold": f"{name} ({m['threshold']:.3f})", "recall": m["sensitivity"],
                 "precision": m["precision"], "specificity": m["specificity"], "accuracy": m["accuracy"],
                 "balanced_acc": m["balanced_accuracy"], "F1": m["f1"], "TP": tp, "FP": fp, "TN": tn, "FN": fn})
display(pd.DataFrame(rows).set_index("threshold").round(3))

## 7. Overfitting check
For each fold/seed run: losses at the best epoch (kept by early stopping) and at the last epoch trained.
Overfitting shows as training loss falling while validation loss rises after the best epoch.

In [ ]:
h = pd.read_csv(f"results/experiments/{EXPERIMENT}/history.csv")
rows = []
for (fold, seed), g in h.groupby(["fold", "seed"]):
    best, last = g.loc[g["inner_val_loss"].idxmin()], g.iloc[-1]
    rows.append({"fold": fold, "seed": seed, "best_epoch": int(best.epoch), "last_epoch": int(last.epoch),
                 "train_loss_best": best.train_loss, "train_loss_last": last.train_loss,
                 "val_loss_best": best.inner_val_loss, "val_loss_last": last.inner_val_loss})
r = pd.DataFrame(rows)
display(r.round(3))
print("Mean:", r.drop(columns=["fold", "seed"]).mean().round(3).to_dict())

## 8. Held-out test set (scored once)
If the test set has already been scored for this experiment, the saved report is shown instead of re-running it.

In [ ]:
import os, json
report = f"results/experiments/{EXPERIMENT}/test_report.json"
if not os.path.exists(report):
    !python -m densenet_t2d.final_test --experiment {EXPERIMENT}
t = json.load(open(report))
m = t["at_threshold"]["dev_selected"]
(tn, fp), (fn, tp) = m["confusion_matrix"]
print(f"TEST  n={t['n']}  T2D={t['n_pos']}  prevalence {t['prevalence']:.3f}")
print(f"ROC-AUC {t['roc_auc']:.3f} (95% CI {t['roc_auc_ci95'][0]:.3f}-{t['roc_auc_ci95'][1]:.3f})   [CV was {t['dev_oof_auc']:.3f}]")
print(f"AP      {t['average_precision']:.3f} (95% CI {t['average_precision_ci95'][0]:.3f}-{t['average_precision_ci95'][1]:.3f})")
print(f"At threshold {m['threshold']:.3f}: recall {m['sensitivity']:.3f} | specificity {m['specificity']:.3f} | "
      f"precision {m['precision']:.3f} | NPV {tn / (tn + fn):.3f} | accuracy {m['accuracy']:.3f} | F1 {m['f1']:.3f}")
print(f"TP={tp}  FP={fp}  TN={tn}  FN={fn}")

## 9. Figure: learning curves, ROC, precision–recall, confusion matrix

In [ ]:
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

EXP = f"results/experiments/{EXPERIMENT}"
hist = pd.read_csv(f"{EXP}/history.csv")
oof  = pd.read_csv(f"{EXP}/oof.csv")
test = pd.read_csv(f"{EXP}/test_predictions.csv")
thr  = json.load(open(f"{EXP}/summary.json"))["selected_threshold"]

fig, ax = plt.subplots(2, 2, figsize=(13, 10))
BLUE, ORANGE, GREY = "#2563eb", "#ea580c", "#9ca3af"

a = ax[0, 0]
for _, g in hist.groupby(["fold", "seed"]):
    a.plot(g.epoch, g.train_loss, color=BLUE, alpha=0.12)
    a.plot(g.epoch, g.inner_val_loss, color=ORANGE, alpha=0.12)
m = hist.groupby("epoch")[["train_loss", "inner_val_loss"]].mean()
a.plot(m.index, m.train_loss, color=BLUE, lw=2.5, label="Training loss (mean)")
a.plot(m.index, m.inner_val_loss, color=ORANGE, lw=2.5, label="Validation loss (mean)")
a.set(title="Learning curves (5 folds × 3 seeds)", xlabel="Epoch", ylabel="Weighted BCE loss")
a.legend(); a.grid(alpha=0.3)

a = ax[0, 1]
for df, name, c in [(oof, "Cross-validation", BLUE), (test, "Test", ORANGE)]:
    fpr, tpr, _ = roc_curve(df.label, df.prob)
    a.plot(fpr, tpr, color=c, lw=2.5, label=f"{name} (n={len(df)})  AUC {roc_auc_score(df.label, df.prob):.3f}")
a.plot([0, 1], [0, 1], "--", color=GREY, label="Chance  AUC 0.500")
pred = (test.prob >= thr).astype(int)
sens = ((pred == 1) & (test.label == 1)).sum() / (test.label == 1).sum()
spec = ((pred == 0) & (test.label == 0)).sum() / (test.label == 0).sum()
a.scatter(1 - spec, sens, s=90, color="black", zorder=5, label=f"Test operating point (thr {thr:.3f})")
a.set(title="ROC curve", xlabel="False positive rate (1 − specificity)", ylabel="True positive rate (sensitivity)")
a.legend(loc="lower right"); a.grid(alpha=0.3)

a = ax[1, 0]
for df, name, c in [(oof, "Cross-validation", BLUE), (test, "Test", ORANGE)]:
    p, r, _ = precision_recall_curve(df.label, df.prob)
    a.plot(r, p, color=c, lw=2.5, label=f"{name}  AP {average_precision_score(df.label, df.prob):.3f}")
a.axhline(test.label.mean(), ls="--", color=GREY, label=f"Chance (test prevalence {test.label.mean():.2f})")
a.set(title="Precision–recall curve", xlabel="Recall (sensitivity)", ylabel="Precision", ylim=(0, 1))
a.legend(); a.grid(alpha=0.3)

a = ax[1, 1]
cm = np.array([[((pred == 0) & (test.label == 0)).sum(), ((pred == 1) & (test.label == 0)).sum()],
               [((pred == 0) & (test.label == 1)).sum(), ((pred == 1) & (test.label == 1)).sum()]])
a.imshow(cm, cmap="Blues")
names = [["True negative", "False positive"], ["False negative", "True positive"]]
for i in range(2):
    for j in range(2):
        a.text(j, i, f"{names[i][j]}\n{cm[i, j]}", ha="center", va="center", fontsize=13,
               color="white" if cm[i, j] > cm.max() / 2 else "black")
a.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Predicted: no T2D", "Predicted: T2D"],
      yticklabels=["Actual: no T2D", "Actual: T2D"], title=f"Test confusion matrix (threshold {thr:.3f})")

fig.suptitle("DenseNet-121 (TorchXRayVision, lr 3e-4): T2D detection from chest X-rays", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{EXP}/final_model_figure.png", dpi=200, bbox_inches="tight")
plt.show()